# Part 9 — Comparative Evaluation, Robustness, and Trade-offs

This notebook answers the assignment's central evaluation questions: Which system performs best? Why? What does it cost? Where does it remain weak?

**Scope:** all scores are stored development measurements. The 60 locked test questions remain unused.


In [1]:
from collections import Counter
from pathlib import Path
import csv
import hashlib
import html
import json
import platform
import random
import statistics

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SEED = 20250816
random.seed(SEED)
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

def load_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))

def print_table(rows, columns):
    if not rows:
        print("(no rows)")
        return
    widths = {
        column: max(len(str(column)), *(len(str(row.get(column, ""))) for row in rows))
        for column in columns
    }
    print(" | ".join(str(column).ljust(widths[column]) for column in columns))
    print("-+-".join("-" * widths[column] for column in columns))
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))

def write_bar_svg(filename, values, title, *, maximum=None):
    values = list(values)
    width, left, right, row_height = 820, 245, 80, 34
    height = 76 + row_height * len(values)
    plot_width = width - left - right
    largest = maximum or max((float(value) for _, value in values), default=1.0) or 1.0
    elements = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        f'<text x="{width / 2}" y="27" text-anchor="middle" font-family="Arial" font-size="18" font-weight="700">{html.escape(title)}</text>',
    ]
    for index, (label, value) in enumerate(values):
        y = 52 + index * row_height
        bar_width = plot_width * float(value) / largest
        elements.extend([
            f'<text x="{left - 10}" y="{y + 17}" text-anchor="end" font-family="Arial" font-size="13">{html.escape(str(label))}</text>',
            f'<rect x="{left}" y="{y}" width="{bar_width:.2f}" height="20" rx="3" fill="#1c5b58"/>',
            f'<text x="{min(left + bar_width + 7, width - 58):.2f}" y="{y + 16}" font-family="Arial" font-size="12">{float(value):.4g}</text>',
        ])
    elements.append('</svg>')
    target = FIGURES / filename
    target.write_text("\n".join(elements) + "\n", encoding="utf-8")
    print(f"Saved visualization: {target.relative_to(ROOT)}")
    return target

print(f"Project: {ROOT.name} | Python: {platform.python_version()} | fixed seed: {SEED}")


Project: h | Python: 3.12.13 | fixed seed: 20250816


## Complete performance comparison


In [2]:
baselines = load_json("reports/tables/baselines_development.json")
bridge = load_json("reports/tables/querybridge_development.json")
reranker = load_json("reports/tables/reranker_depth20.json")
title = load_json("reports/tables/application_accuracy_regression.json")
assert {baselines["test_queries_used"], bridge["test_queries_used"], reranker["test_queries_used"], title["test_queries_used"]} == {0}
systems = []
for name, values in baselines["systems"].items():
    systems.append((name, values, values["mean_latency_ms"]))
systems.extend([("QueryBridge + RRF", bridge["result"], bridge["result"]["mean_latency_ms"]), ("QueryBridge + depth-20 reranker", reranker["after_reranking"], reranker["mean_retrieval_ms"] + reranker["mean_rerank_ms"]), ("Current pipeline + title route", title["after"], 580.8)])
rows = [{"system": name, "R@1": f'{values["recall_at_1"]:.4f}', "R@5": f'{values["recall_at_5"]:.4f}', "R@10": f'{values["recall_at_10"]:.4f}', "MRR@10": f'{values["mrr_at_10"]:.4f}', "nDCG@10": f'{values["ndcg_at_10"]:.4f}', "mean_ms": f'{latency:.1f}'} for name, values, latency in systems]
print_table(rows, ["system", "R@1", "R@5", "R@10", "MRR@10", "nDCG@10", "mean_ms"])
write_bar_svg("all_systems_recall_at_10.svg", [(name, values["recall_at_10"]) for name, values, _ in systems], "All systems: development Recall@10", maximum=1.0)


system                          | R@1    | R@5    | R@10   | MRR@10 | nDCG@10 | mean_ms
--------------------------------+--------+--------+--------+--------+---------+--------
direct_dense                    | 0.0500 | 0.0833 | 0.0917 | 0.0615 | 0.0686  | 51.8   
single_transliteration_bm25     | 0.0000 | 0.0167 | 0.0250 | 0.0074 | 0.0116  | 60.8   
standard_hybrid                 | 0.0250 | 0.0667 | 0.0917 | 0.0456 | 0.0565  | 94.5   
QueryBridge + RRF               | 0.0500 | 0.1000 | 0.1667 | 0.0750 | 0.0959  | 444.0  
QueryBridge + depth-20 reranker | 0.1250 | 0.1750 | 0.1833 | 0.1444 | 0.1540  | 16947.6
Current pipeline + title route  | 0.3917 | 0.8750 | 0.9833 | 0.5830 | 0.6796  | 580.8  
Saved visualization: reports\figures\all_systems_recall_at_10.svg


![All systems Recall at 10](../reports/figures/all_systems_recall_at_10.svg)


**Best measured retrieval system:** the current pipeline with romanized-title matching. It reaches 0.9833 Recall@10, 0.5830 MRR@10, and 0.6796 nDCG@10 before reranking. It wins because the diagnostic task is strongly title/entity-oriented and character n-grams bridge noisy Roman spellings to concise Urdu titles. This is a development retrieval result, not unrestricted answer accuracy.


## Robustness before the title-route change


In [3]:
robustness = load_json("reports/tables/robustness.json")
assert robustness["test_queries_used"] == 0
category_rows = []
for query_type, results in sorted(robustness["query_types"].items()):
    category_rows.append({"query_type": query_type, "n": results["direct_dense"]["queries"], "dense_R@10": f'{results["direct_dense"]["recall_at_10"]:.4f}', "bridge_R@10": f'{results["querybridge_no_reranker"]["recall_at_10"]:.4f}', "reranked_R@10": f'{results["querybridge_reranked"]["recall_at_10"]:.4f}'})
print_table(category_rows, ["query_type", "n", "dense_R@10", "bridge_R@10", "reranked_R@10"])
write_bar_svg("robustness_recall_at_10.svg", [(row["query_type"], float(row["reranked_R@10"])) for row in category_rows], "Original reranked pipeline: Recall@10 by query type", maximum=0.55)


query_type                  | n  | dense_R@10 | bridge_R@10 | reranked_R@10
----------------------------+----+------------+-------------+--------------
abbreviated_roman_urdu      | 15 | 0.0000     | 0.0000      | 0.0000       
clean_roman_urdu            | 16 | 0.0625     | 0.3125      | 0.5000       
highly_noisy_roman_urdu     | 15 | 0.0667     | 0.2667      | 0.2667       
informal_spelling           | 16 | 0.0000     | 0.1250      | 0.0625       
named_entity                | 14 | 0.0000     | 0.0000      | 0.0714       
short_query                 | 15 | 0.1333     | 0.3333      | 0.2667       
slightly_ambiguous          | 15 | 0.0667     | 0.1333      | 0.0667       
urdu_english_code_switching | 14 | 0.4286     | 0.1429      | 0.2143       
Saved visualization: reports\figures\robustness_recall_at_10.svg


![Robustness by query type](../reports/figures/robustness_recall_at_10.svg)


The original reranked system performs best on clean questions (0.5000 Recall@10) and fails completely on abbreviated questions. Named entities reach only 0.0714. This breakdown explains why aggregate semantic retrieval was producing unrelated results and directly motivates title matching. A category-wise post-title-route run is not available, so it would be incorrect to replace this table with inferred values.


## Resource and latency analysis


In [4]:
resources = load_json("reports/tables/latency_resources.json")
latency = resources["query_latency_ms"]
latency_rows = [{"system": name, "mean_ms": values["mean"], "p95_ms": values.get("p95", "not recorded")} for name, values in latency.items()]
print_table(latency_rows, ["system", "mean_ms", "p95_ms"])
print("Embedding shape:", resources["embedding_matrix_shape"])
print("Embedding index MB:", round(resources["embedding_index_bytes"] / (1024 ** 2), 2))
print("Cold index build minutes:", round(resources["index_build_seconds"] / 60, 2))
print("Peak observed resident memory MB:", resources["peak_observed_resident_memory_mb"])
write_bar_svg("latency_comparison.svg", [(row["system"], float(row["mean_ms"])) for row in latency_rows], "Mean CPU query latency (ms)")


system                            | mean_ms   | p95_ms      
----------------------------------+-----------+-------------
direct_dense                      | 51.788    | 61.666      
querybridge_no_reranker           | 444.043   | 656.212     
querybridge_plus_reranker_depth20 | 16947.604 | not recorded
reranker_depth20                  | 16397.043 | not recorded
single_transliteration_bm25       | 60.758    | 101.098     
standard_hybrid                   | 94.522    | 127.279     
Embedding shape: [16352, 384]
Embedding index MB: 23.95
Cold index build minutes: 42.44
Peak observed resident memory MB: 2550.64
Saved visualization: reports\figures\latency_comparison.svg


![Latency comparison](../reports/figures/latency_comparison.svg)


## Trade-off discussion and remaining limitations

- The depth-20 cross-encoder improves early rank ordering but adds roughly 16.4 seconds per query and about 2.5 GB observed resident memory. It is optional rather than the default interactive path.
- The title route offers the best measured quality/latency balance at roughly 581 ms before reranking, but it is structurally suited to title-definition questions.
- The 4,000-article corpus cannot answer every Urdu information need, especially current prices or breaking news.
- The diagnostic set is small, constructed, and has one recorded relevant passage per query. Alternative valid evidence may be under-counted.
- The high Recall@10 result shows candidate coverage only. Independent end-to-end answer review, category-wise re-evaluation, calibration, and the one-time locked-test run remain necessary.
